# Data Cleaning


In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import KFold
import hashlib
import re 
raw_data_path= '../Data/Raw Data/'
clean_data_path= '../Data/Clean Data/'
raw_data_file=raw_data_path+'raw_data.csv'
df=pd.read_csv(raw_data_file)

In [2]:
# Rename columns for convenience
column_names={
    'TRANSACTION_NUMBER':'transaction_num',
    'INSTANCE_DATE':'instance_date',
    'GROUP_EN':'group',
    'PROCEDURE_EN':'procedure',
    'IS_OFFPLAN_EN':'is_offplan',
    'IS_FREE_HOLD_EN':'is_free_hold',
    'USAGE_EN':'usage',
    'AREA_EN':'area',
    'PROP_TYPE_EN':'prop_type',
    'PROP_SB_TYPE_EN':'prop_sb_type',
    'TRANS_VALUE':'trans_value',
    'PROCEDURE_AREA':'procedure_area',
    'ACTUAL_AREA':'actual_area',
    'ROOMS_EN':'rooms',
    'PARKING':'parking',
    'NEAREST_METRO_EN':'nearest_metro',
    'NEAREST_MALL_EN':'nearest_mall',
    'NEAREST_LANDMARK_EN':'nearest_landmark',
    'TOTAL_BUYER':'total_buyer',
    'TOTAL_SELLER':'total_seller',
    'MASTER_PROJECT_EN':'master_project',
    'PROJECT_EN':'project'
}
df.rename(columns=column_names,inplace=True)

#Hash transaction number
df['transaction_num']=df['transaction_num'].apply(lambda x:hashlib.sha256(str(x).encode()).hexdigest())
#Drop requried columns
df=df.drop(columns=['total_buyer','total_seller','master_project'])

In [3]:
before = len(df)
df = df[df['group'] == 'Sales'].copy()
print(f"After Sales-only filter: {len(df):,} rows ({before - len(df):,} dropped)")

# Filter to Unit property type — Land and Building have different price functions.
# Mixing them forces the model to learn three distributions at once.
before = len(df)
df = df[df['prop_type'] == 'Unit'].copy()
print(f"After Unit-only filter: {len(df):,} rows ({before - len(df):,} dropped)")

# Drop exact duplicates
before = len(df)
df = df.drop_duplicates().reset_index(drop=True)
print(f"After dedupe: {len(df):,} rows ({before - len(df):,} dropped)")

After Sales-only filter: 71,923 rows (21,955 dropped)
After Unit-only filter: 59,242 rows (12,681 dropped)
After dedupe: 59,242 rows (0 dropped)


In [4]:
#Turn date datatype to datetime
df['instance_date']=pd.to_datetime(df['instance_date'])
df['date_year']= df['instance_date'].dt.year
#Months turned into sin and cos vectors
df['month_sin']=np.sin(2 * np.pi*df['instance_date'].dt.month/12.0)
df['month_cos']=np.cos(2*np.pi*df['instance_date'].dt.month/12.0)

In [8]:
#Calculate the baseline price per square foot
df['price_per_sqft']=df['trans_value']/df['actual_area']

#Sort chronologically within each community to properly calculate rolling windows
df=df.sort_values(by=['area','instance_date']).reset_index(drop=True)

#Set the date as the index for time based rolling math
df=df.set_index('instance_date')

#Generate the rolling 30 day average price per square foot for each specific area
df['community_30d_avg_price']=df.groupby('area')['price_per_sqft'].transform(lambda x:x.shift(1).rolling('30D').mean())

df['_offplan_flag'] = (df['is_offplan'] == 'Off-Plan').astype(int)
df['off_plan_vol_ratio'] = (df.groupby('area')['_offplan_flag'].transform(lambda x: x.shift(1).rolling('30D', min_periods=1).mean()))
df = df.drop(columns=['_offplan_flag'])

#Reset the index and drop the raw timestamp now that temporal features are extracted
df.reset_index(inplace=True)

df['community_30d_avg_price'] = df.groupby('area')['community_30d_avg_price'].transform(
    lambda x: x.fillna(x.mean())
)
df['off_plan_vol_ratio'] = df.groupby('area')['off_plan_vol_ratio'].transform(
    lambda x: x.fillna(x.mean())
)
# Areas with only one transaction → no group mean → fill with global mean
df['community_30d_avg_price'] = df['community_30d_avg_price'].fillna(df['price_per_sqft'].mean())
df['off_plan_vol_ratio'] = df['off_plan_vol_ratio'].fillna(0.5)

In [9]:
#Clean up rooms values
df['rooms']=df['rooms'].astype(str).str.extract(r'(\d+)').fillna(0).astype(int)

#Clean parking by finding first number found. if not found return nan
def clean_parking(val):
    if pd.isna(val):
        return np.nan
    s = str(val).strip()
    nums = re.findall(r'\d+', s)
    if not nums:
        return 1 if s else np.nan
    if re.match(r'^[A-Za-z]+[-\s]\d+$', s):
        return 1
    total = sum(int(n) for n in nums)
    return min(total, 10)
df['parking'] = df['parking'].apply(clean_parking)

In [10]:
df['is_offplan'] = df['is_offplan'].map({'Off-Plan': 1, 'Ready': 0}).fillna(0).astype(int)
df['is_free_hold'] = df['is_free_hold'].map({'Free Hold': 1, 'Non Free Hold': 0}).fillna(0).astype(int)
df['usage'] = df['usage'].map({'Residential': 1, 'Commercial': 0}).fillna(1).astype(int)

In [11]:
#Fill categorical columns with Unknown
cat_cols_with_na=['prop_sb_type','nearest_metro','nearest_mall','nearest_landmark','project']
df[cat_cols_with_na]=df[cat_cols_with_na].fillna('Unknown')
#Fill numerical columns with median
num_cols_with_na=['rooms','parking']
for col in num_cols_with_na:
    df[col]=df[col].fillna(df[col].median())

In [12]:
#Remove rows with negative or zero area/value
df = df[df['trans_value']>0]
df = df[df['actual_area']>0]
df = df[df['procedure_area']>0]

#Remove extreme top 1% outliers for numerical features
for col in ['trans_value','actual_area','procedure_area']:
    q01=df[col].quantile(0.01)
    q99=df[col].quantile(0.99)
    df = df[(df[col] >= q01) & (df[col] <= q99)]
df['price_per_sqft'] = df['trans_value'] / df['actual_area']
df = df[(df['price_per_sqft'] >= 4000) & (df['price_per_sqft'] <= 100000)].reset_index(drop=True)

print(f"After outlier removal: {len(df):,} rows")

After outlier removal: 55,743 rows


In [13]:
# procedure_area and actual_area are 97% identical — keep only one to avoid 
# splitting SHAP attribution across collinear features.
df = df.drop(columns=['procedure_area'])

# Group rare procedures into "Other" so we don't explode to 30 one-hot columns
proc_counts = df['procedure'].value_counts()
rare_procs = proc_counts[proc_counts < 50].index
df['procedure'] = df['procedure'].where(~df['procedure'].isin(rare_procs), 'Other')

# 🔴 Out-of-fold target encoding for 'area' (was: full-data mean, which leaks the target).
# Each row's encoding is computed from OTHER folds' price_per_sqft, not its own.
def oof_target_encode(df, col, target, n_splits=5, smoothing=10):
    global_mean = df[target].mean()
    encoded = pd.Series(index=df.index, dtype=float)
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)
    for train_idx, val_idx in kf.split(df):
        agg = df.iloc[train_idx].groupby(col)[target].agg(['mean', 'count'])
        # Bayesian smoothing — small-sample areas pulled toward global mean
        smoothed = (agg['mean'] * agg['count'] + global_mean * smoothing) / (agg['count'] + smoothing)
        encoded.iloc[val_idx] = df.iloc[val_idx][col].map(smoothed).fillna(global_mean)
    return encoded

df['area'] = oof_target_encode(df, 'area', 'price_per_sqft')

# Apply OOF target encoding to other high-cardinality features too 
# (LabelEncoder imposes a fake ordinal relationship, which is bad for trees)
for col in ['project', 'prop_sb_type', 'nearest_metro', 'nearest_mall', 'nearest_landmark']:
    df[col] = oof_target_encode(df, col, 'price_per_sqft')

# One-hot the now-small 'procedure' column. Drop 'group' and 'prop_type' 
# since they're constant after the Sales+Unit filter.
df = df.drop(columns=['group', 'prop_type'])
df = pd.get_dummies(df, columns=['procedure'], drop_first=True)

In [14]:
df = df.sort_values('instance_date').reset_index(drop=True)
# Now drop instance_date (cyclical/year features carry the temporal signal)
df = df.drop(columns=['instance_date'])
df.to_csv(clean_data_path + 'cleaned_data.csv', index=False)

In [15]:
dfc=pd.read_csv(clean_data_path+'cleaned_data.csv')
dfc.head()
for col in dfc.columns:
    print(f"{col}: {len(df[col].tolist())}")


transaction_num: 55743
is_offplan: 55743
is_free_hold: 55743
usage: 55743
area: 55743
prop_sb_type: 55743
trans_value: 55743
actual_area: 55743
rooms: 55743
parking: 55743
nearest_metro: 55743
nearest_mall: 55743
nearest_landmark: 55743
project: 55743
date_year: 55743
month_sin: 55743
month_cos: 55743
price_per_sqft: 55743
community_30d_avg_price: 55743
off_plan_vol_ratio: 55743
procedure_Delayed Sell: 55743
procedure_Development Registration: 55743
procedure_Development Registration Pre-Registration: 55743
procedure_Lease to Own Registration: 55743
procedure_Other: 55743
procedure_Sale: 55743
procedure_Sale On Payment Plan: 55743
procedure_Sell - Pre registration: 55743
procedure_Sell Development: 55743
